# LibTorch Bucket DataLoader - Proxy Benchmark (COMPARABLE)

Reorganized so results are **directly comparable** to the category-scheduler
pipeline (`sft_final`). Comparability requires ALL of these to match:

| must match | value |
|---|---|
| datasets | same sources (Path A real / Path B synthetic) |
| tokenizer | `inceptionai/jais-family-590m` |
| max-seq-len | `1024` |
| batch size | `32` |
| proxy constants | same D / A / C from calibration |

**Proxy model (identical in both pipelines):**

`time = D*batches + A*padded_tokens + C*sum(B*T^2)`

fed via `PROXY_BETA=D`, `PROXY_ALPHA=A`, `PROXY_GAMMA=C`. Using the *same*
calibrated constants means the only thing being measured is the batching strategy.

## 0 - Setup: HF/Xet workaround, clone, deps

In [10]:
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_XET_DISABLE"]     = "1"
os.environ["HF_ENDPOINT"]        = "https://huggingface.co"

import subprocess, sys
subprocess.run([sys.executable,"-m","pip","uninstall","-y","hf_xet","hf-xet"],
               capture_output=True)
try:
    import hf_xet; print("hf_xet STILL INSTALLED - restart runtime, run this cell first")
except ModuleNotFoundError:
    print("hf_xet gone")

hf_xet gone


In [11]:
import os, sys, subprocess, glob, shutil, json

REPO = "https://github.com/SniAssia/Optimized_data_loaders.git"
shutil.rmtree("/content/Optimized_data_loaders", ignore_errors=True)
subprocess.run(["git","clone","--depth","1",REPO,"/content/Optimized_data_loaders"], check=True)
CODE = "/content/Optimized_data_loaders/sft_libtorch_proxy"
assert os.path.isfile(os.path.join(CODE,"libtorch_dataloader.h")), "repo layout unexpected"
print("code:", CODE)

import importlib
need = [m for m in ["torch","transformers","datasets"] if importlib.util.find_spec(m) is None]
if need: subprocess.run([sys.executable,"-m","pip","install","-q",*need], check=True)
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
TORCH_CMAKE = torch.utils.cmake_prefix_path

code: /content/Optimized_data_loaders/sft_libtorch_proxy
torch 2.11.0+cu128 | CUDA: True


## 1 - Patch: add the quadratic term to the proxy

The category pipeline's per-batch model is `D + A*(B*T) + C*(B*T^2)`. Summed over
an epoch that is `D*batches + A*padded_tokens + C*sum(B*T^2)`. The repo already
has the first two terms (`beta`, `alpha`); this cell adds the `sum(B*T^2)`
accumulator and `PROXY_GAMMA` so BOTH pipelines evaluate the identical formula.
Idempotent - safe to re-run.

In [12]:
h = os.path.join(CODE, "libtorch_dataloader.h")
s = open(h).read()

if "total_attn_units" in s:
    print("already patched")
else:
    # 1) accumulator field
    s = s.replace(
        "    std::atomic<int64_t> total_real_tokens{0};",
        "    std::atomic<int64_t> total_real_tokens{0};\n"
        "    std::atomic<int64_t> total_attn_units{0};   // sum of B*T^2 (quadratic term)",
        1)
    # 2) reset
    s = s.replace("        total_real_tokens = 0;",
                  "        total_real_tokens = 0;\n        total_attn_units  = 0;", 1)
    # 3) accumulate per batch
    s = s.replace("        total_real_tokens += real_tokens;",
                  "        total_real_tokens += real_tokens;\n"
                  "        total_attn_units  += B * L * L;", 1)
    # 4) gamma in the proxy block
    s = s.replace(
        '            double alpha = 1.0, beta = 0.0;',
        '            double alpha = 1.0, beta = 0.0, gamma = 0.0;', 1)
    s = s.replace(
        '            if (const char* b = std::getenv("PROXY_BETA"))  beta  = std::atof(b);',
        '            if (const char* b = std::getenv("PROXY_BETA"))  beta  = std::atof(b);\n'
        '            if (const char* g = std::getenv("PROXY_GAMMA")) gamma = std::atof(g);', 1)
    s = s.replace(
        '            double proxy   = alpha * static_cast<double>(padded)\n'
        '                           + beta  * static_cast<double>(total_batches.load());',
        '            double proxy   = alpha * static_cast<double>(padded)\n'
        '                           + beta  * static_cast<double>(total_batches.load())\n'
        '                           + gamma * static_cast<double>(total_attn_units.load());', 1)
    # 5) print batches + attn units + hours so the notebook can parse them
    s = s.replace(
        '            std::cout << "  Proxy train time     : " << std::fixed << proxy\n'
        '                      << "  (alpha=" << alpha << ", beta=" << beta << ")\\n";',
        '            std::cout << "  Batches              : " << total_batches.load() << "\\n";\n'
        '            std::cout << "  Attn units sum BT2   : " << total_attn_units.load() << "\\n";\n'
        '            std::cout << "  Proxy train time (s) : " << std::fixed << proxy\n'
        '                      << "  (alpha=" << alpha << ", beta=" << beta\n'
        '                      << ", gamma=" << gamma << ")\\n";\n'
        '            std::cout << "  Proxy train time (h) : " << proxy/3600.0 << "\\n";', 1)
    open(h,"w").write(s)
    print("patched: total_attn_units + PROXY_GAMMA")

# verify every edit landed
chk = open(h).read()
for k in ["total_attn_units{0}", "total_attn_units  = 0;", "total_attn_units  += B * L * L;",
          "PROXY_GAMMA", "gamma * static_cast<double>(total_attn_units.load())"]:
    print(("OK  " if k in chk else "MISS"), k)

patched: total_attn_units + PROXY_GAMMA
OK   total_attn_units{0}
OK   total_attn_units  = 0;
OK   total_attn_units  += B * L * L;
OK   PROXY_GAMMA
OK   gamma * static_cast<double>(total_attn_units.load())


## 2 - Build

In [13]:
p = os.path.join(CODE, "tokenize_dataset.py")
s = open(p).read()

if "--limit" in s:
    print("already patched")
else:
    # 1) argparse: per-source --limit, aligned with --input order
    s = s.replace(
        '''    parser.add_argument("--split",  default="train")''',
        '''    parser.add_argument("--limit", action="append", default=None,
                        help="Max records for the corresponding --input "
                             "(same order). Use 0 / omit for no cap.")
    parser.add_argument("--split",  default="train")''', 1)

    # 2) cap helper + per-source limit map, wrapping make_gen
    s = s.replace(
        '''    def make_gen(inp):
        if inp.endswith(".jsonl") or os.path.isfile(inp):
            print(f"[tokenize] source: local JSONL → {inp}")
            return stream_jsonl(inp)
        print(f"[tokenize] source: HuggingFace → {inp} (split={args.split})")
        return stream_hf_dataset(inp, args.split,
                                 args.instruction_col, args.response_col)''',
        '''    def _capped(gen, n):
        """Yield at most n records (n<=0 => no cap)."""
        if not n or n <= 0:
            yield from gen
            return
        for i, rec in enumerate(gen):
            if i >= n:
                print(f"[tokenize] limit {n:,} reached — stopping this source")
                break
            yield rec

    # per-source caps aligned to --input order; missing entries => no cap
    _limits = [int(x) for x in (args.limit or [])]
    _limits += [0] * (len(args.input) - len(_limits))
    LIMITS = dict(zip(args.input, _limits))
    for _i, _l in LIMITS.items():
        print(f"[tokenize] cap: {_i} -> {('none' if _l <= 0 else format(_l, ','))}")

    def make_gen(inp, cap=None):
        if cap is None:
            cap = LIMITS.get(inp, 0)
        if inp.endswith(".jsonl") or os.path.isfile(inp):
            print(f"[tokenize] source: local JSONL → {inp}")
            return _capped(stream_jsonl(inp), cap)
        print(f"[tokenize] source: HuggingFace → {inp} (split={args.split})")
        return _capped(stream_hf_dataset(inp, args.split,
                                         args.instruction_col, args.response_col), cap)''', 1)

    # 3) stats preview must NOT eat into the cap
    s = s.replace("        preview_gen = make_gen(args.input[0])",
                  "        preview_gen = make_gen(args.input[0], cap=0)", 1)

    open(p, "w").write(s)
    print("patched: per-source --limit added")

chk = open(p).read()
for k in ['parser.add_argument("--limit"', 'def _capped(gen, n)',
          'LIMITS = dict(zip', 'make_gen(inp, cap=None)']:
    print(("OK  " if k in chk else "MISS"), k)

patched: per-source --limit added
OK   parser.add_argument("--limit"
OK   def _capped(gen, n)
OK   LIMITS = dict(zip
OK   make_gen(inp, cap=None)


In [14]:
p = os.path.join(CODE, "tokenize_dataset.py")
s = open(p).read()

if 'SPLITS = dict(zip' in s:
    print("already patched")
else:
    s = s.replace(
        '''    parser.add_argument("--split",  default="train")''',
        '''    parser.add_argument("--split", action="append", default=None,
                        help="Split for the corresponding --input (same order). "
                             "Defaults to 'train' for any source not given one.")''', 1)

    s = s.replace(
        '''    def make_gen(inp, cap=None):''',
        '''    _splits = list(args.split or [])
    _splits += ["train"] * (len(args.input) - len(_splits))
    SPLITS = dict(zip(args.input, _splits))
    for _i, _sp in SPLITS.items():
        print(f"[tokenize] split: {_i} -> {_sp}")

    def make_gen(inp, cap=None):''', 1)

    s = s.replace(
        '''        print(f"[tokenize] source: HuggingFace → {inp} (split={args.split})")
        return _capped(stream_hf_dataset(inp, args.split,
                                         args.instruction_col, args.response_col), cap)''',
        '''        _sp = SPLITS.get(inp, "train")
        print(f"[tokenize] source: HuggingFace → {inp} (split={_sp})")
        return _capped(stream_hf_dataset(inp, _sp,
                                         args.instruction_col, args.response_col), cap)''', 1)
    open(p, "w").write(s)
    print("patched: per-source --split")

patched: per-source --split


In [21]:
p = os.path.join(CODE, "tokenize_dataset.py")
s = open(p).read()

OLD = '''    was_truncated = False

    if not prompt_ids or prompt_ids[-1] != tok.eos_id:
        prompt_ids.append(tok.eos_id)
    if not resp_ids or resp_ids[-1] != tok.eos_id:
        resp_ids.append(tok.eos_id)

    if len(prompt_ids) > max_seq_len:
        prompt_ids    = prompt_ids[:max_seq_len - 1] + [tok.eos_id]
        was_truncated = True
    if len(resp_ids) > max_seq_len:
        resp_ids      = resp_ids[:max_seq_len - 1] + [tok.eos_id]
        was_truncated = True'''

NEW = '''    was_truncated = False

    if not prompt_ids or prompt_ids[-1] != tok.eos_id:
        prompt_ids.append(tok.eos_id)
    if not resp_ids or resp_ids[-1] != tok.eos_id:
        resp_ids.append(tok.eos_id)

    # BUDGET-AWARE TRUNCATION. The old code capped prompt and response
    # INDEPENDENTLY at max_seq_len, so a sample could leave here with
    # 1024+1024=2048 tokens. The C++ collate then did the real truncation via
    #   r_actual = min(response_len, batch_max_len - p_actual)
    # which silently DROPS THE WHOLE RESPONSE when a long prompt eats the
    # budget -> that sample carries no loss. Rule now: response is never
    # truncated unless it alone exceeds the budget; the prompt yields first;
    # prompt+response <= max_seq_len is guaranteed here so the collate-time
    # squeeze can never fire.
    PROMPT_KEEP = os.environ.get("PROMPT_TRUNC_KEEP", "tail")   # "tail" | "head"

    if len(prompt_ids) + len(resp_ids) > max_seq_len:
        if len(resp_ids) > max_seq_len:
            resp_ids      = resp_ids[:max_seq_len - 1] + [tok.eos_id]
            was_truncated = True
        budget = max_seq_len - len(resp_ids)
        if len(prompt_ids) > budget:
            if budget <= 1:
                prompt_ids = [tok.eos_id]
            elif PROMPT_KEEP == "head":
                prompt_ids = prompt_ids[:budget - 1] + [tok.eos_id]
            else:
                prompt_ids = prompt_ids[-(budget - 1):] + [tok.eos_id]
            was_truncated = True

    assert len(prompt_ids) + len(resp_ids) <= max_seq_len, \\
        f"budget violated: {len(prompt_ids)}+{len(resp_ids)} > {max_seq_len}"'''

if "BUDGET-AWARE TRUNCATION" in s:
    print("already patched")
else:
    assert OLD in s, "pattern not found"
    open(p, "w").write(s.replace(OLD, NEW, 1))
    print("patched: budget-aware truncation (response protected)")

patched: budget-aware truncation (response protected)


In [15]:
if subprocess.run(["which","cmake"], capture_output=True).returncode != 0:
    subprocess.run([sys.executable,"-m","pip","install","-q","cmake"], check=True)

build = os.path.join(CODE, "build")
shutil.rmtree(build, ignore_errors=True); os.makedirs(build, exist_ok=True)
cfg = subprocess.run(["cmake", f"-DCMAKE_PREFIX_PATH={TORCH_CMAKE}",
                      "-DCMAKE_BUILD_TYPE=Release", ".."],
                     cwd=build, capture_output=True, text=True)
print(cfg.stdout[-800:])
if cfg.returncode:
    print("CMAKE STDERR:\n", cfg.stderr[-2500:]); raise RuntimeError("configure failed")

mk = subprocess.run(["cmake","--build",".","--target","dataloader_libtorch","-j","2"],
                    cwd=build, capture_output=True, text=True)
print(mk.stdout[-1200:])
if mk.returncode:
    print("BUILD STDERR:\n", mk.stderr[-3000:]); raise RuntimeError("build failed")
BIN = os.path.join(build, "dataloader_libtorch")
print("built:", os.path.exists(BIN))

lkit directory: /usr/local/cuda
-- PyTorch: Header version is: 12.8
-- Found Python: /usr/local/bin/python (found version "3.12.13") found components: Interpreter
-- /usr/local/cuda/lib64/libnvrtc.so shorthash is 2bb82d1a
-- USE_CUDNN is set to 0. Compiling without cuDNN support
-- USE_CUSPARSELT is set to 0. Compiling without cuSPARSELt support
-- USE_CUDSS is set to 0. Compiling without cuDSS support
-- USE_CUFILE is set to 0. Compiling without cuFile support
-- Autodetected CUDA architecture(s):  7.5
-- Added CUDA NVCC flags for: -gencode;arch=compute_75,code=sm_75
-- Found Torch: /usr/local/lib/python3.12/dist-packages/torch/lib/libtorch.so
-- Configuring done (5.6s)
-- Generating done (0.0s)
-- Build files have been written to: /content/Optimized_data_loaders/sft_libtorch_proxy/build

[ 50%] Building CXX object CMakeFiles/dataloader_libtorch.dir/main_libtorch.cpp.o
[100%] Linking CXX executable dataloader_libtorch
[100%] Built target dataloader_libtorch

built: True


## 3 - Shared parameters (MUST equal the category pipeline)

Paste the D / A / C you got from `calibrate_A_C()` in the sft_final notebook.
If these differ, the hour numbers are not comparable.

In [16]:
TOKENIZER   = "inceptionai/jais-family-590m"   # same as sft_final
MAX_SEQ_LEN = 1024                             # same
SHARD_SIZE  = 1024                             # same
BATCH       = 32                               # same as pc.B

# --- calibrated constants from the category pipeline. PASTE YOURS. ---
D_INTERCEPT = 0.0604      # sec/batch    -> PROXY_BETA
A_LINEAR    = 2.791e-05   # sec/token    -> PROXY_ALPHA
C_QUAD      = 2.361e-08   # sec/token^2  -> PROXY_GAMMA
print(f"proxy: t = {D_INTERCEPT}*batches + {A_LINEAR:.3e}*padded + {C_QUAD:.3e}*sum(B*T^2)")

proxy: t = 0.0604*batches + 2.791e-05*padded + 2.361e-08*sum(B*T^2)


---
# PATH A - REAL datasets

The same three sources the category pipeline used. Requires an HF token.
Run **either** Path A or Path B, then go to section 4.

In [17]:
from google.colab import userdata
from huggingface_hub import login
HF_TOKEN = userdata.get("HF_TOKEN")
login(HF_TOKEN); print("HF authenticated")

HF authenticated


In [18]:
DATA = os.path.join(CODE, "data_real")
cmd = [sys.executable, "tokenize_dataset.py",
       "--input", "tatsu-lab/alpaca",             "--split", "train",     "--limit", "0",
       "--input", "HuggingFaceH4/ultrachat_200k", "--split", "train_sft", "--limit", "0",
       "--input", "Open-Orca/OpenOrca",           "--split", "train",     "--limit", "200000",
       "--output-dir", DATA,
       "--tokenizer", TOKENIZER,
       "--max-seq-len", str(MAX_SEQ_LEN),
       "--shard-size", str(SHARD_SIZE),
       "--skip-stats"]

env = dict(os.environ)
env["HF_TOKEN"] = HF_TOKEN; env["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
env["HF_HUB_DISABLE_XET"] = "1"
r = subprocess.run(cmd, cwd=CODE, capture_output=True, text=True, env=env)
print(r.stdout[-2000:])
if r.returncode:
    print("STDERR:\n", r.stderr[-2500:]); raise RuntimeError("tokenize failed")
man = json.load(open(os.path.join(DATA,"manifest.json")))
print("manifest: max_seq_len", man["max_seq_len"], "| shards", len(man["shards"]))

orch_proxy/data_real/shard_439 (53 truncated)
[ShardFlusher] shard_439 written (1024 records)
  [shard] wrote 1024 samples → /content/Optimized_data_loaders/sft_libtorch_proxy/data_real/shard_440 (46 truncated)
[ShardFlusher] shard_440 written (1024 records)
  [shard] wrote 1024 samples → /content/Optimized_data_loaders/sft_libtorch_proxy/data_real/shard_441 (46 truncated)
[ShardFlusher] shard_441 written (1024 records)
  [shard] wrote 1024 samples → /content/Optimized_data_loaders/sft_libtorch_proxy/data_real/shard_442 (49 truncated)
[ShardFlusher] shard_442 written (1024 records)
  [shard] wrote 1024 samples → /content/Optimized_data_loaders/sft_libtorch_proxy/data_real/shard_443 (45 truncated)
[ShardFlusher] shard_443 written (1024 records)
  [shard] wrote 1024 samples → /content/Optimized_data_loaders/sft_libtorch_proxy/data_real/shard_444 (45 truncated)
[ShardFlusher] shard_444 written (1024 records)
  [shard] wrote 1024 samples → /content/Optimized_data_loaders/sft_libtorch_proxy

---
# PATH B - SYNTHETIC datasets (no network)

Regenerates the **byte-identical** JSONL files (same seed 1234 as
`sft_final/gen_datasets.py`), so both pipelines consume exactly the same samples.
This is the most rigorous comparison: identical input, only the loader differs.

In [ ]:
import json, random
random.seed(1234)                      # SAME seed as sft_final/gen_datasets.py
SRC = os.path.join(CODE, "data_synth_src"); os.makedirs(SRC, exist_ok=True)

WORDS = ("the quick brown fox jumps over lazy dog while data flows through "
         "pipelines and models learn patterns from tokens across many batches "
         "with attention layers computing weighted sums of hidden states during "
         "training steps that optimize loss over gradients and parameters").split()
INSTRUCTIONS = ["Explain the following concept in detail:",
                "Summarize the text below:",
                "Answer the question:",
                "Translate the passage:",
                "Write a short analysis of:",
                "Continue the following story:",
                "Describe step by step how to:"]

def make_text(n): return " ".join(random.choice(WORDS) for _ in range(max(1, n)))

def emit(path, n, choices):
    with open(path, "w", encoding="utf-8") as f:
        for _ in range(n):
            iw, ow = random.choice(choices)
            rec = {"instruction": random.choice(INSTRUCTIONS) + " " + make_text(iw),
                   "input": "", "output": make_text(ow)}
            f.write(json.dumps(rec) + "\n")
    print("wrote", path, n)

SHORT, MEDIUM, LONG, XLONG = (10,30), (60,180), (250,500), (700,1100)
emit(os.path.join(SRC,"ds_small_60k.jsonl"),  60000, [SHORT]*7 + [MEDIUM]*3)
emit(os.path.join(SRC,"ds_tiny_20k.jsonl"),   20000, [SHORT]*2 + [MEDIUM]*5 + [LONG]*3)
emit(os.path.join(SRC,"ds_big_200k.jsonl"),  200000, [SHORT]*4 + [MEDIUM]*3 + [LONG]*2 + [XLONG]*1)

In [ ]:
DATA = os.path.join(CODE, "data_synth")
cmd = [sys.executable, "tokenize_dataset.py",
       "--input", os.path.join(SRC,"ds_small_60k.jsonl"),
       "--input", os.path.join(SRC,"ds_tiny_20k.jsonl"),
       "--input", os.path.join(SRC,"ds_big_200k.jsonl"),
       "--output-dir", DATA,
       "--tokenizer", TOKENIZER,
       "--max-seq-len", str(MAX_SEQ_LEN),
       "--shard-size", str(SHARD_SIZE),
       "--skip-stats"]
r = subprocess.run(cmd, cwd=CODE, capture_output=True, text=True)
print(r.stdout[-2000:])
if r.returncode:
    print("STDERR:\n", r.stderr[-2500:]); raise RuntimeError("tokenize failed")
man = json.load(open(os.path.join(DATA,"manifest.json")))
print("manifest: max_seq_len", man["max_seq_len"], "| shards", len(man["shards"]))

---
# 4 - Run the benchmark (calibrated -> seconds)

In [19]:
import re
MANIFEST = os.path.join(DATA, "manifest.json")
env = dict(os.environ,
           PROXY_ALPHA=str(A_LINEAR),
           PROXY_BETA=str(D_INTERCEPT),
           PROXY_GAMMA=str(C_QUAD))
r = subprocess.run([BIN, MANIFEST, str(BATCH)], cwd=CODE, env=env,
                   capture_output=True, text=True)
print(r.stdout[-4000:])
if r.returncode: print("STDERR:\n", r.stderr[-1500:])

def grab(pat):
    m = re.search(pat, r.stdout)
    return float(m.group(1).replace(",", "")) if m else None

libtorch = {
    "padded_tokens": grab(r"Padded tokens\s*:\s*([\d,]+)"),
    "real_tokens":   grab(r"Real \(useful\) tokens\s*:\s*([\d,]+)"),
    "padding_pct":   grab(r"Padding %\s*:\s*([\d.]+)"),
    "batches":       grab(r"Batches\s*:\s*([\d,]+)"),
    "proxy_hours":   grab(r"Proxy train time \(h\)\s*:\s*([\d.]+)"),
}
print("\nparsed:", libtorch)

d_183  disk=0.6ms  parse=0.5ms  disk=1.8MB  ram=1.8MB  records=1024
[ShardCache] shard_134  disk=0.7ms  parse=0.5ms  disk=1.6MB  ram=1.7MB  records=1024
[ShardCache] shard_156  disk=0.6ms  parse=0.7ms  disk=1.6MB  ram=1.7MB  records=1024
[ShardCache] shard_19  disk=0.5ms  parse=0.5ms  disk=1.1MB  ram=1.2MB  records=1024
[ShardCache] shard_273  disk=0.8ms  parse=0.7ms  disk=1.7MB  ram=1.8MB  records=1024
[ShardCache] shard_68  disk=0.5ms  parse=0.4ms  disk=1.2MB  ram=1.3MB  records=1024
[ShardCache] shard_397  disk=0.8ms  parse=0.5ms  disk=1.8MB  ram=1.9MB  records=1024
[ShardCache] shard_365  disk=0.8ms  parse=0.6ms  disk=1.8MB  ram=1.9MB  records=1024
[ShardCache] shard_127  disk=0.5ms  parse=0.6ms  disk=1.6MB  ram=1.6MB  records=1024
[ShardCache] shard_212  disk=0.6ms  parse=0.9ms  disk=1.7MB  ram=1.8MB  records=1024
[ShardCache] shard_191  disk=0.7ms  parse=0.6ms  disk=1.7MB  ram=1.7MB  records=1024

  BATCH STATISTICS
  Total batches    : 14370
  Total samples    : 459839
  Total t

## 5 - Comparison table

Paste the category pipeline's `estimate_epoch_seconds()` summary below.

**Validity checklist** - the comparison is only meaningful if both runs used:
same datasets, same tokenizer, same max-seq-len, same batch size, same D/A/C.

In [20]:
# ---- paste from the sft_final notebook summary ----
category = {
    "padded_tokens": 291_861_704,
    "real_tokens":   261_066_844,
    "padding_pct":   10.55,
    "batches":       16_552,
    "proxy_hours":   3.66,
}
# ---------------------------------------------------

rows = [("LibTorch bucketing", libtorch), ("Category scheduler", category)]
hdr = f"{'method':22s} {'padded tokens':>15s} {'padding %':>10s} {'batches':>9s} {'proxy h':>9s}"
print(hdr); print("-"*len(hdr))
for name, m in rows:
    print(f"{name:22s} {int(m['padded_tokens'] or 0):>15,} "
          f"{m['padding_pct'] or 0:>9.2f}% {int(m['batches'] or 0):>9,} "
          f"{m['proxy_hours'] or 0:>9.2f}")

if libtorch["padded_tokens"] and category["padded_tokens"]:
    dp = 100*(libtorch["padded_tokens"]-category["padded_tokens"])/libtorch["padded_tokens"]
    dh = 100*(libtorch["proxy_hours"]-category["proxy_hours"])/libtorch["proxy_hours"]
    print(f"\ncategory vs bucketing: {dp:+.1f}% padded tokens, {dh:+.1f}% proxy time")
    print("(negative = category scheduler wins)")

    # sanity: real tokens should be ~identical if the data really is the same
    if libtorch["real_tokens"] and category["real_tokens"]:
        drift = abs(libtorch["real_tokens"]-category["real_tokens"])/category["real_tokens"]
        print(f"\nreal-token drift: {100*drift:.2f}%  "
              + ("(OK - same data)" if drift < 0.02 else
                 "(WARNING: >2% - the two runs did NOT see the same samples!)"))

method                   padded tokens  padding %   batches   proxy h
---------------------------------------------------------------------
LibTorch bucketing         461,191,360     60.75%    14,370      6.86
Category scheduler         291,861,704     10.55%    16,552      3.66

category vs bucketing: +36.7% padded tokens, +46.6% proxy time
(negative = category scheduler wins)

real-token drift: 30.67%  (WARNING: >2% - the two runs did NOT see the same samples!)
